<a href="https://colab.research.google.com/github/raunakraj1310/SaginaBolo/blob/main/HindiNMTASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dependencies

In [ ]:
# 1. Install dependencies
!pip -q install -U transformers torchaudio soundfile gradio huggingface_hub onnxruntime

print("✅ Dependencies installed")


In [ ]:
import os
import gradio as gr
import numpy as np
import torch
import librosa
import sys
import time
import json
import csv
import traceback
from pathlib import Path
from datetime import datetime, timezone
import transformers
import huggingface_hub

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Xet disabled:", os.environ["HF_HUB_DISABLE_XET"])

In [ ]:
# 0. Verify Colab GPU



print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime → Change runtime type → GPU"
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    f"VRAM: "
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)

In [ ]:

from huggingface_hub import login
login()

# Transcription

In [ ]:
def transcribe(audio):

# Load an audio file
  wav, sr = torchaudio.load(audio)
  wav = torch.mean(wav, dim=0, keepdim=True)

  target_sample_rate = 16000  # Expected sample rate
  if sr != target_sample_rate:
      resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
      wav = resampler(wav)


  # Perform ASR with RNNT decoding
  transcription_rnnt = model(wav, "hi", "rnnt")
  return(transcription_rnnt)


# IndicTrans2 HF Inference

We provide an example notebook on how to use our IndicTrans2 models which were originally trained with the fairseq to HuggingFace transformers for inference purpose.

## Setup

Please run the cells below to install the necessary dependencies.

In [ ]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [ ]:
%%capture
%cd /content/IndicTrans2/huggingface_interface

In [ ]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

**IMPORTANT : Restart your run-time first and then run the cells below.**

## ASR

In [ ]:
import os
import gradio as gr
import numpy as np
import torch
import librosa
import sys
import time
import json
import csv
import traceback
from pathlib import Path
from datetime import datetime, timezone
import transformers
import huggingface_hub

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None

In [ ]:
from transformers import AutoModel
import torch, torchaudio

# Load the model
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)



In [ ]:
def transcribe(audio):

# Load an audio file
  wav, sr = torchaudio.load(audio)
  wav = torch.mean(wav, dim=0, keepdim=True)

  target_sample_rate = 16000  # Expected sample rate
  if sr != target_sample_rate:
      resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
      wav = resampler(wav)


  # Perform ASR with RNNT decoding
  transcription_rnnt = model(wav, "hi", "rnnt")
  return(transcription_rnnt)


In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

# Translation


In [ ]:
indic_en_ckpt_dir = "ai4bharat/indictrans2-indic-en-1B"  # ai4bharat/indictrans2-indic-en-dist-200M
indic_en_tokenizer, indic_en_model = initialize_model_and_tokenizer(indic_en_ckpt_dir, quantization)

ip = IndicProcessor(inference=True)


In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

In [ ]:
def translate(transcript,src_lang="hin_Deva", tgt_lang="eng_Latn"):
  en_translations = batch_translate([transcript], src_lang, tgt_lang, indic_en_model, indic_en_tokenizer, ip)
  print(f"\n{src_lang} - {tgt_lang}")
  return (en_translations)
  # for input_sentence, translation in zip([transcript], en_translations):
  #     print(f"{src_lang}: {input_sentence}")
  #     print(f"{tgt_lang}: {translation}")

In [ ]:
text="""
उस्ताद राशिद ख़ान को एक संगीत-प्रेमी कैसे याद कर सकता है? इस पर ठहरता हूँ तो कुछ तस्वीरें ज़ेहन में आती हैं। ‘राग यमन’, ‘मारवा’, ‘सोहनी’, ‘मेघ’ और ‘ललित’ जैसे गंभीर ख़याल गाने वाले सिद्ध गायक—राशिद ख़ान। जहाँ भारतीय शास्त्रीय संगीत का बेहद सधा हुआ अनुशासन मौजूद है। जैसे एक समर्पित साधक अपने साधना में लीन हो। उप शास्त्रीय संगीत ‘ठुमरी’, पूर्वी, ‘टप्पे’ और ‘तराना’ इत्यादि गाने वाले— राशिद ख़ान। जहाँ एक आत्मीय खिलंदड़ापन और नोक-झोंक मौजूद हो। सभी संगतकारों को जगह देता हुआ, संवादी और लोकतांत्रिक— राशिद ख़ान। फ़िल्मी संगीत, जैज और कोक स्टूडियो जैसे युवाओं में लोकप्रिय गीत गाने वाले— राशिद ख़ान। एक परंपराबद्ध आधुनिक। जिसके पास कुछ नई और लोकप्रिय चीज़ों के लिए भी अवकाश है। यह अवकाश कुछ उस क़िस्म का है जहाँ नई पीढ़ी से एक ऐसा संवाद की कोई सूरत निकले। यह कुछ-कुछ सुनने की तैयारी कराने जैसा हो या एक कोशिश कि इस रास्ते कुछ नए श्रोताओं को जोड़ा जा सके। संकटमोचन मंदिर के परिसर में टहलते हुए अपने विचार में मग्न एक आस्तिक इंसान। अपने पुत्र को संकटमोचन मंदिर में संकटमोचन के सामने सजदा करने के लिए उत्साहित करता हुआ एक आत्मीय पिता। बनारस में गंगा घाट पर भीड़ से अपने को बचा कर टहलते हुए एक आम मनुष्य और संकटमोचन मंदिर में कार्यक्रम से पहले सुरमंडल मिलाते और कार्यक्रम के बाद अपने मुरीदों से मिलते हुए— राशिद ख़ान। ये वे छवियाँ हैं जिनसे मुझे राशिद ख़ान की याद आती है।
कला में छप्पन वर्ष की उम्र कोई बहुत बड़ी उम्र नहीं मानी जाती। यह कला के पकने की उम्र होती है। कला में निखार आने में एक लंबा वक़्त लगता है। यह राशिद ख़ान साहेब की ख़ुशनसीबी थी कि उन्हें कला जगत में वह पहचान और प्रसिद्धि दोनों मिली जिसके वह सबल हकदार थे। भारत रत्न पंडित भीमसेन जोशी ने राशिद ख़ान को ‘भारतीय शास्त्रीय संगीत का भविष्य’ कहा था।

‘आश्वस्ति’ कुछ अजीब शै है। मृत्यु अक्सर वह छोड़ जाती है, जिसमें हम जीवन के चिह्नों को तलाशते हैं। किंचित भारतीय श्रोताओं की यह बदनसीबी ही कही जाएगी कि अक्सर हमारे भविष्य को बहुत कम समय मिलता है। हम ऐसे त्रासद और हतभाग्य समय में जीने को विवश हैं, जहाँ जेनुइन प्रतिभाएँ उँगलियों पर गिनी जा सकती हैं। यह ऐसा समय है जहाँ कम मनुष्यों की भरमार है। बौनों और आत्ममुग्धों की संख्या हर क्षेत्र में बढती गई है। उस्ताद राशिद ख़ान के जाने से जो रिक्ति हुई है, इसकी भरपाई अब दूर तक संभव प्रतीत नहीं होती दिख रही है।
उस्ताद राशिद ख़ान रामपुर सहसवान घराने के ध्वजवाहक कलाकार। एक कलाकार की तैयारी और प्रस्तुति को अक्सर एक श्रोता समझ नहीं पाता। उसे एक उत्पाद के मानिंद एक तैयार चीज़ मिलती है। मेरा भी उस्ताद राशिद ख़ान से कुछ ऐसा ही संबंध रहा है—उत्पादक और भोक्ता का, किंतु संबंध चाहे जैसा हो एक समय के बाद उससे मोह होना स्वभाविक मानवीय गुण है। हमारी कल्पनाओं में भी यह रिक्ति कहीं नहीं थी। पिछले दिनों जब वह बीमार हुए तो यह उम्मीद थी कि उनके जीवन की यह मध्य-लय सम पर आकर उनकी ख़याल गायकी की ही तरह द्रुत में आ जाएगी, लेकिन यह ‘पुनः सवेरा एक और फेरा है जी का’ ही साबित हुआ। उन्होंने मध्य-लय के सम को ही अपना अंतिम सम माना।

"""
egtrans=translate(text,"hin_Deva", "eng_Latn")
egtrans

In [ ]:
print(egtrans)

In [ ]:
# src_lang, tgt_lang = "hin_Deva", "eng_Latn"
# transcript=transcribe("/content/OSR_in_000_0063_8k.wav")
# translate(transcript,"hin_Deva", "eng_Latn")

# Implementation

In [ ]:
def live(audio):
  transcripts=transcribe(audio)
  translate(transcripts,src_lang, tgt_lang)


# Record

In [ ]:
model.eval()

In [ ]:
import gradio as gr
import numpy as np
import torch
import torchaudio
from transformers import AutoModel


model.eval()

In [ ]:
model = model.to(DEVICE)
model.eval()

print("Model loaded.")
print("Device:", DEVICE)

In [ ]:
import gradio as gr
print("Gradio version:", gr.__version__)

In [ ]:
def transcribe(audio):
    if audio is None:
        return ""

    # Gradio may provide (sample_rate, numpy_array)
    sr, y = audio

    # Convert stereo -> mono
    if y.ndim > 1:
        y = y.mean(axis=1)

    # Convert to float32
    y = y.astype(np.float32)

    # Avoid division by zero
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))

    # Convert numpy -> torch
    wav = torch.from_numpy(y).unsqueeze(0)

    # Resample to 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        wav = resampler(wav)

    # Move to GPU
    wav = wav.to(DEVICE)

    # Indic-Conformer RNNT decoding
    with torch.no_grad():
        transcription = model(wav, "hi", "rnnt")

    return transcription

# Successful

In [ ]:
import os
import base64
import torch
import torchaudio
import gradio as gr

from IPython.display import Javascript, display
from google.colab.output import eval_js


# ---------------------------------------------------------
# 1. Browser microphone recorder
# ---------------------------------------------------------

def record_from_browser(seconds=5):
    js = Javascript("""
    async function recordAudio(seconds) {

        const stream = await navigator.mediaDevices.getUserMedia({
            audio: true
        });

        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = event => {
            if (event.data.size > 0) {
                chunks.push(event.data);
            }
        };

        recorder.start();

        await new Promise(resolve =>
            setTimeout(resolve, seconds * 1000)
        );

        recorder.stop();

        await new Promise(resolve => {
            recorder.onstop = resolve;
        });

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {
            type: recorder.mimeType
        });

        const reader = new FileReader();

        return await new Promise(resolve => {
            reader.onloadend = () => resolve(reader.result);
            reader.readAsDataURL(blob);
        });
    }

    recordAudio
    """)

    display(js)

    data = eval_js(f"recordAudio({seconds})")

    audio_bytes = base64.b64decode(data.split(",")[1])

    webm_path = "/content/current_recording.webm"

    with open(webm_path, "wb") as f:
        f.write(audio_bytes)

    return webm_path


# ---------------------------------------------------------
# 2. Convert browser recording to 16 kHz mono WAV
# ---------------------------------------------------------

def convert_to_wav(webm_path):

    wav_path = "/content/current_recording.wav"

    waveform, sr = torchaudio.load(webm_path)

    # Stereo -> mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Resample to 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    torchaudio.save(
        wav_path,
        waveform,
        16000
    )

    return wav_path


# ---------------------------------------------------------
# 3. Hindi ASR
# ---------------------------------------------------------

def transcribe_wav(wav_path):

    print("Processing:", wav_path)

    waveform, sr = torchaudio.load(wav_path)

    if waveform.shape[0] > 1:
        waveform = torch.mean(
            waveform,
            dim=0,
            keepdim=True
        )

    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    print("Sample rate:", sr)
    print("Shape:", waveform.shape)

    result = model(
        waveform,
        "hi",
        "rnnt"
    )

    print("Transcription:", result)

    return str(result)



In [ ]:
def record_and_transcribe(seconds):

    print("\n🎤 Recording...")

    # Record audio
    webm_path = record_from_browser(seconds)

    print("Recorded:", webm_path)
    print("Size:", os.path.getsize(webm_path), "bytes")

    # Convert to WAV
    wav_path = convert_to_wav(webm_path)

    print("Converted:", wav_path)

    # -----------------------------------------
    # STEP 1: Complete Hindi transcription
    # -----------------------------------------

    final_transcription = transcribe_wav(wav_path)

    print("\n🇮🇳 FINAL HINDI:")
    print(final_transcription)

    # -----------------------------------------
    # STEP 2: Translate ONLY after ASR is done
    # -----------------------------------------

    english_translation = translate(
        final_transcription,
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    print("\n🇬🇧 ENGLISH:")
    print(english_translation)

    # Return both
    return final_transcription, english_translation, wav_path

In [ ]:
def transcribe_wav(wav_path, chunk_seconds=8):
    print("\n🎧 Loading:", wav_path)

    waveform, sr = torchaudio.load(wav_path)

    # Stereo → mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Resample → 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    sr = 16000

    total_samples = waveform.shape[1]
    total_seconds = total_samples / sr

    print(f"🎵 Total duration: {total_seconds:.2f} seconds")

    chunk_samples = int(chunk_seconds * sr)

    transcriptions = []

    for start in range(0, total_samples, chunk_samples):

        end = min(start + chunk_samples, total_samples)

        chunk = waveform[:, start:end]

        print(
            f"\n🧩 Processing "
            f"{start/sr:.1f}s → {end/sr:.1f}s"
        )

        if chunk.shape[1] < int(0.3 * sr):
            continue

        result = model(
            chunk,
            "hi",
            "rnnt"
        )

        result = str(result).strip()

        print("📝 Chunk:", result)

        if result:
            transcriptions.append(result)

    final_transcription = " ".join(transcriptions)

    print("\n==============================")
    print("FINAL TRANSCRIPTION:")
    print(final_transcription)
    print("==============================")

    return final_transcription

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 🎤 Hindi Speech → English Translation
        """
    )

    duration = gr.Slider(
        minimum=2,
        maximum=60,
        value=10,
        step=1,
        label="Recording duration (seconds)"
    )

    record_button = gr.Button("🎙️ Record Hindi")

    transcription = gr.Textbox(
        label="🇮🇳 Hindi Transcription",
        lines=6
    )

    translation = gr.Textbox(
        label="🇬🇧 English Translation",
        lines=6
    )

    recorded_audio = gr.Audio(
        label="Recorded Audio",
        type="filepath"
    )

    record_button.click(
        fn=record_and_transcribe,
        inputs=duration,
        outputs=[
            transcription,
            translation,
            recorded_audio
        ]
    )

demo.launch(debug=True)